In [1]:
import pandas as pd
import numpy as np

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [2]:
data_gdynia = pd.read_csv(
    "../../data/raw/srodmiescie_raw.csv", sep=";", index_col=0, dtype=column_types
).dropna(subset="ground_floor_function")

In [3]:
data_gdynia["date_of_checking"] = pd.to_datetime(
    data_gdynia["date_of_checking"], format="%d.%m.%Y"
)

In [4]:
data_clean = data_gdynia.assign(
    ground_floor_function=lambda d: (
        d["ground_floor_function"].str.replace("_", " ").str.lower()
    )
)

In [5]:
street_fixes = {
    "Hryniewickiego": "Antoniego Hryniewickiego",
    "Wendy": "Tadeusza Wendy",
    "Plac Kaszubski - Jana z Kolna": "Jana z Kolna",
}

data_clean["addr_street"] = data_clean["addr_street"].replace(street_fixes)

data_clean["addr_number"] = data_clean["addr_number"].str.strip().str.upper()

In [6]:
style_fixes = {
    "nowożytnośc": "współczesność",
    "współczsnośc": "współczesność", 
    "wspóółczesność": "współczesność",
    "wspóczesność": "współczesność",
    "współczesnosc": "współczesność",
    "traycjonalizm": "tradycjonalizm",
    "wiejski": "tradycjonalizm",
    "art. deco": "modernizm",
    "artdeco": "modernizm",
    "wertykalizm": "modernizm",
    "nowożytność" : "współczesność",
}

data_clean["arch_style"] = data_clean["arch_style"].str.lower().str.strip().replace(style_fixes)

In [7]:
data_long = (
    data_clean.assign(
        ground_floor_function=lambda d: d["ground_floor_function"].str.split(",")
    )
    .explode("ground_floor_function")
    .assign(ground_floor_function=lambda d: d["ground_floor_function"].str.strip())
)
data_long = data_long.loc[:, ["addr_street", "ground_floor_function"]]


In [8]:
num_func_buil = data_long.groupby(level=0).agg(
    number_of_functions=("ground_floor_function", "count")
)

data_clean = pd.merge(
    data_clean, num_func_buil, left_index=True, right_index=True, how="outer"
)

In [9]:
data_clean.to_csv(
    "../../data/processed/fixed/srodmiescie.csv", sep=";", encoding="utf-8"
)
data_long.to_csv(
    "../../data/processed/fixed/srodmiescie_long.csv", sep=";", encoding="utf-8"
)

In [10]:
data_clean

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking,number_of_functions
id,,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,hotel,NaN,Hotel,7,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
2,890729731,Świętego Piotra,6D,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
3,890292040,Węglowa,22,NaN,centrum handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
4,890729732,Świętego Piotra,6,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
5,136885205,Tadeusza Wendy,15,NaN,centrum medyczne,NaN,Centrum Medyczne,6,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,2,2026-09-27,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1237,126180359,Władysława IV,61,NaN,"laboratorium medyczne,piekarnia,kiosk",NaN,Mieszkaniowa Wielorodzinna,7,<NA>,1940,modernizm,3,NaN,NaN,google.com/maps,2,2026-09-30,3
1238,126173503,Świętojańska,122,NaN,"apteka,handel zdrowie",NaN,Biurowa,6,<NA>,1930,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-30,2
1239,260676369,Aleja Marszałka Józefa Piłsudskiego,56,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,7,<NA>,1930,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-30,1


In [11]:
category_map = pd.read_csv(
    "../../data/reference/category_map.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)

In [12]:
data_long_categories = pd.merge(
    data_long,
    category_map,
    left_on="ground_floor_function",
    right_index=True,
    indicator=True,
    how="outer",
    validate="many_to_one",
)

In [13]:
data_long_export = data_long_categories.loc[:,['addr_street','function_clean','category']]
data_long_export.to_csv('../../data/processed/fixed/srodmiescie_long_categories.csv', sep=";", encoding="utf-8")